# Lab 05: Generative Models and Vision-Language Models

In this lab, you will explore three important families of models in modern machine learning:

1. **CLIP (Contrastive Language-Image Pre-Training)** — a vision-language model that learns joint image-text embeddings
2. **Variational Autoencoders (VAEs)** — a generative model that learns a latent representation of data
3. **Diffusion Models** — a generative model that learns to reverse a gradual noising process


Please fill in all code cells marked with `TODO` comments.

## Setup

Run the cell below to install required packages.

In [ ]:
# Install required packages
!pip install -q transformers diffusers accelerate torch torchvision datasets pillow matplotlib numpy seaborn

# If using gated models (e.g., Stable Diffusion), you may need to authenticate:
# !huggingface-cli login

---
## Part 1: CLIP (Contrastive Language-Image Pre-Training)

**CLIP** is a model developed by OpenAI that learns visual concepts from natural language supervision. It is trained on a large dataset of image-text pairs using a **contrastive learning** objective: matching images are pulled together in the shared embedding space while non-matching pairs are pushed apart.

Key ideas:
- An **image encoder** (Vision Transformer or ResNet) maps images to embedding vectors.
- A **text encoder** (Transformer) maps text descriptions to embedding vectors.
- The model is trained so that cosine similarity between matching image-text pairs is maximized and non-matching pairs is minimized.
- This enables powerful **zero-shot** capabilities: CLIP can classify images into categories it has never been explicitly trained on, simply by comparing image embeddings to text embeddings of class descriptions.

We will use the `openai/clip-vit-base-patch32` model from HuggingFace throughout this section.

### Question 1.1: Loading CLIP and Preprocessing

In this question, you will:
1. Load the CLIP model (`CLIPModel`) and processor (`CLIPProcessor`) from HuggingFace Transformers using the checkpoint `openai/clip-vit-base-patch32`.
2. Load the CIFAR-10 test set (first 200 samples) using the HuggingFace `datasets` library.
3. Display 5 random sample images along with their class labels.

CIFAR-10 classes: `['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']`

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import CLIPModel, CLIPProcessor
from datasets import load_dataset
import random

# CIFAR-10 class names
cifar10_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

# TODO: Load the CLIP model and processor from 'openai/clip-vit-base-patch32'
model = torch.load("CLIPModel")
processor = CLIPProcessor  # TODO

# TODO: Load the CIFAR-10 test set (first 200 samples)
# Hint: load_dataset("cifar10", split="test[:200]")
cifar10 = None  # TODO

# TODO: Select 5 random indices and display the corresponding images with their labels
# Use matplotlib to create a 1x5 subplot grid
# Access images via cifar10[i]['img'] and labels via cifar10[i]['label']


### Question 1.2: Zero-Shot Image Classification

Use CLIP to perform **zero-shot classification** on 100 CIFAR-10 test images. The idea is:
1. Create text prompts of the form `"a photo of a {class_name}"` for each of the 10 CIFAR-10 classes.
2. For each image, compute the cosine similarity between the image embedding and each text embedding.
3. The predicted class is the one with the highest similarity.
4. Calculate and print the overall accuracy.

You should process images in batches for efficiency.

In [ ]:
# TODO: Create text prompts for zero-shot classification
text_prompts = [f"a photo of a {cls}" for cls in cifar10_classes]

# TODO: Encode all text prompts using the CLIP processor and model
# Hint: Use processor(text=text_prompts, return_tensors="pt", padding=True)
# Then pass through model.get_text_features()


# TODO: For 100 CIFAR-10 test images, compute image embeddings
# Hint: Use processor(images=..., return_tensors="pt")
# Then pass through model.get_image_features()


# TODO: Compute cosine similarity between each image embedding and all text embeddings
# Hint: Normalize embeddings, then use matrix multiplication


# TODO: Determine predicted classes (argmax of similarity scores)


# TODO: Compute and print accuracy
# Compare predicted labels to ground truth labels from the dataset


### Question 1.3: Custom Prompt Engineering

The choice of text prompt significantly affects CLIP's zero-shot classification performance. In this question, you will:

1. Experiment with **at least 3 different prompt templates** for the CIFAR-10 classes. Examples:
   - `"a photo of a {class_name}"`
   - `"a blurry photo of a {class_name}"`
   - `"a bright photo of a {class_name}"`
   - `"a centered photo of a {class_name}"`
   - `"a low resolution photo of a {class_name}"`
   - `"a pixelated image of a {class_name}"`
   
2. Compute the zero-shot classification accuracy for each prompt template on the same 100 images.
3. Create a **bar chart** comparing accuracy across prompt templates.

In [ ]:
# TODO: Define at least 3 different prompt templates
prompt_templates = [
    "a photo of a {}",
    # TODO: Add more prompt templates
]

# TODO: For each prompt template, compute zero-shot accuracy on 100 images
# Store results in a dictionary: {template_string: accuracy}
accuracies = {}

for template in prompt_templates:
    # TODO: Create text prompts using this template for all 10 classes
    # TODO: Encode texts, encode images, compute similarities
    # TODO: Compute accuracy and store in accuracies dict
    pass

# TODO: Create a bar chart showing accuracy per prompt template
# Use matplotlib, label axes appropriately, and rotate x-tick labels if needed


### Question 1.4: Image-Text Similarity Heatmap

In this question, you will visualize how CLIP relates images to text descriptions:

1. Select **8 images** from different CIFAR-10 classes.
2. Create **8 text descriptions** — include a mix of matching and non-matching descriptions (e.g., some that correctly describe the image and some that do not).
3. Compute the **similarity matrix** (8x8) between all image and all text embeddings.
4. Visualize the matrix as a **heatmap** using `matplotlib`, with proper axis labels showing which images and texts correspond to each row/column.

In [ ]:
# TODO: Select 8 images from different CIFAR-10 classes
# Find indices in the dataset where each class appears and pick one image per class (for 8 classes)
selected_images = []  # TODO: list of PIL images
selected_labels = []  # TODO: list of string labels

# TODO: Create 8 text descriptions (mix of matching and non-matching)
text_descriptions = [
    # TODO: e.g., "a photo of a cat", "a red sports car", "a bird flying in the sky", etc.
]

# TODO: Compute image embeddings for the 8 selected images


# TODO: Compute text embeddings for the 8 text descriptions


# TODO: Compute the 8x8 cosine similarity matrix


# TODO: Visualize as a heatmap
# Use plt.imshow() or plt.pcolormesh()
# Add colorbar, set x-tick labels to text descriptions, y-tick labels to image labels
# Rotate x-tick labels for readability


---
## Part 2: Variational Autoencoder (VAE)

A **Variational Autoencoder (VAE)** is a generative model that learns a latent representation of the data. Unlike a standard autoencoder, the VAE imposes a probabilistic structure on the latent space:

- The **encoder** maps input $x$ to the parameters of a latent distribution: mean $\mu$ and log-variance $\log \sigma^2$.
- The **reparameterization trick** allows backpropagation through the stochastic sampling: $z = \mu + \sigma \cdot \epsilon$, where $\epsilon \sim \mathcal{N}(0, I)$.
- The **decoder** maps $z$ back to the data space to produce a reconstruction $\hat{x}$.
- The loss function combines:
  - **Reconstruction loss** (e.g., Binary Cross-Entropy): measures how well the decoder reconstructs the input.
  - **KL Divergence**: regularizes the latent space to be close to a standard normal distribution $\mathcal{N}(0, I)$.

$$\mathcal{L} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - D_{KL}(q(z|x) \| p(z))$$

We will also briefly explore **normalizing flows**, which extend the VAE by transforming the simple latent distribution through a series of invertible transformations to obtain a more expressive posterior.

### Question 2.1: Build a VAE

Implement a VAE in PyTorch for MNIST (28x28 grayscale images).

**Architecture:**
- **Encoder:** Input (784) → FC (256) → ReLU → two parallel heads: $\mu$ (16) and $\log\sigma^2$ (16)
- **Decoder:** Latent (16) → FC (256) → ReLU → FC (784) → Sigmoid

You need to implement:
1. `encode(x)`: Pass input through the encoder to get $\mu$ and $\log\sigma^2$
2. `reparameterize(mu, logvar)`: Implement the reparameterization trick
3. `decode(z)`: Pass latent vector through the decoder
4. `forward(x)`: Full forward pass (flatten → encode → reparameterize → decode)
5. `vae_loss(recon_x, x, mu, logvar)`: Compute total loss with BCE reconstruction loss and KL divergence

**KL Divergence (closed form):**
$$D_{KL} = -\frac{1}{2} \sum(1 + \log\sigma^2 - \mu^2 - \sigma^2)$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=16):
        super(VAE, self).__init__()
        # Encoder layers
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        # Decoder layers
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        # TODO: Pass x through fc1 with ReLU, then compute mu and logvar
        pass

    def reparameterize(self, mu, logvar):
        # TODO: Implement the reparameterization trick
        # Sample epsilon from N(0,1), compute z = mu + std * epsilon
        pass

    def decode(self, z):
        # TODO: Pass z through fc3 with ReLU, then fc4 with sigmoid
        pass

    def forward(self, x):
        # TODO: Flatten input, encode, reparameterize, decode
        # Return reconstructed x, mu, and logvar
        pass

def vae_loss(recon_x, x, mu, logvar):
    # TODO: Compute reconstruction loss (BCE) and KL divergence
    # BCE = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')
    # KL = -0.5 * sum(1 + logvar - mu^2 - exp(logvar))
    # Return total loss, reconstruction loss, and KL loss separately
    pass

### Question 2.2: Train the VAE

Train your VAE on the MNIST dataset:
1. Load the MNIST training and test sets with `ToTensor()` transform.
2. Train the VAE for **10 epochs** using the **Adam optimizer** with learning rate `1e-3`.
3. Track the total loss, reconstruction loss, and KL divergence loss per epoch.
4. **Plot** the training loss curves (all three components on the same plot).
5. Show **original vs. reconstructed images** for 10 test samples side by side.

In [ ]:
# TODO: Set device (cuda if available, else cpu)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

# TODO: Load MNIST training and test datasets
# Use transforms.ToTensor() and DataLoader with batch_size=128
# Remember to set download=True when loading MNIST
train_loader = None  # TODO
test_loader = None  # TODO

# TODO: Initialize the VAE model and move to device
vae = None  # TODO

# TODO: Set up Adam optimizer with lr=1e-3
optimizer = None  # TODO

# TODO: Training loop for 10 epochs
# Track: total_losses, recon_losses, kl_losses per epoch
num_epochs = 10
total_losses = []
recon_losses = []
kl_losses = []

for epoch in range(num_epochs):
    # TODO: Set model to train mode
    # TODO: Iterate over batches
    # TODO: Forward pass, compute loss, backprop, optimizer step
    # TODO: Accumulate losses for tracking
    # TODO: Print epoch summary
    pass

# TODO: Plot training loss curves (total, reconstruction, KL)
# Use plt.plot() with labels and legend


# TODO: Show original vs reconstructed images for 10 test samples
# Get a batch from test_loader, pass through VAE, display side by side


### Question 2.3: Latent Space Exploration

Explore the VAE's learned latent space:

1. **Random sampling:** Sample 16 random points from the standard normal distribution $\mathcal{N}(0, I)$ in the 16-dimensional latent space and decode them to generate images. Display the generated images in a 4x4 grid.

2. **2D grid visualization:** Fix all latent dimensions to 0 except the first two. Vary dimensions 0 and 1 across a grid from -3 to 3 (e.g., 20x20 grid). Decode each grid point and arrange the generated images into a large mosaic. This visualizes how the first two latent dimensions control the generated output.

In [ ]:
# Set model to eval mode
vae.eval()

# --- Part A: Random Sampling ---
# TODO: Sample 16 random z vectors from N(0, I) with shape (16, 16)
# TODO: Decode the samples using vae.decode()
# TODO: Reshape to (16, 28, 28) and display in a 4x4 grid


# --- Part B: 2D Grid Visualization ---
# TODO: Create a 20x20 grid of values from -3 to 3 for dimensions 0 and 1
# TODO: For each grid point, create a latent vector with dims 0,1 set to grid values
#        and all other dims set to 0
# TODO: Decode each latent vector
# TODO: Arrange the decoded images into a large mosaic image and display it
# Hint: Create a large numpy array of shape (20*28, 20*28) and fill in each 28x28 block


### Question 2.4: Normalizing Flow Enhancement

**Normalizing flows** improve the expressiveness of the VAE's approximate posterior by applying a series of invertible transformations to the latent variable $z$. Instead of the posterior being restricted to a diagonal Gaussian, the flow transforms it into a more flexible distribution.

A **planar flow** applies the transformation:
$$f(z) = z + u \cdot \tanh(w^T z + b)$$

The log-determinant of the Jacobian is:
$$\log |\det \frac{\partial f}{\partial z}| = \log |1 + u^T \cdot h'(w^T z + b) \cdot w|$$

where $h' = 1 - \tanh^2$ is the derivative of tanh.

Your tasks:
1. Complete the `PlanarFlow` module below.
2. Create a flow-enhanced VAE that applies 2-4 planar flow layers after the reparameterization step.
3. Train the flow-enhanced VAE (5 epochs is sufficient).
4. Visually compare random samples from the base VAE vs. the flow-enhanced VAE.

In [ ]:
class PlanarFlow(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.w = nn.Parameter(torch.randn(1, dim))
        self.b = nn.Parameter(torch.randn(1))
        self.u = nn.Parameter(torch.randn(1, dim))

    def forward(self, z):
        # TODO: Implement planar flow transformation
        # f(z) = z + u * tanh(w^T * z + b)
        # Also compute the log determinant of the Jacobian
        # log|det(df/dz)| = log|1 + u^T * h'(w^T*z + b) * w|
        # where h' = 1 - tanh^2 (derivative of tanh)
        # Return transformed z and log_det_jacobian
        pass


class FlowVAE(VAE):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=16, n_flows=4):
        super().__init__(input_dim, hidden_dim, latent_dim)
        # TODO: Create a list of PlanarFlow layers (use nn.ModuleList)
        self.flows = None  # TODO

    def forward(self, x):
        # TODO: Flatten input, encode, reparameterize
        # TODO: Apply flow layers sequentially, accumulating log_det_jacobian
        # TODO: Decode the transformed z
        # Return recon_x, mu, logvar, and total log_det_jacobian
        pass


def flow_vae_loss(recon_x, x, mu, logvar, log_det_jacobian):
    # TODO: Compute the standard VAE loss (BCE + KL)
    # TODO: Subtract the log_det_jacobian from the loss
    # (The flow improves the posterior, reducing the effective KL divergence)
    pass


# TODO: Initialize and train the FlowVAE for 5 epochs
# TODO: Generate 16 random samples from both the base VAE and FlowVAE
# TODO: Display them side by side for visual comparison


---
## Part 3: Diffusion Models

**Diffusion models** are a class of generative models inspired by non-equilibrium thermodynamics. The key idea involves two processes:

1. **Forward (diffusion) process:** Gradually adds Gaussian noise to the data over $T$ timesteps according to a noise schedule $\{\beta_t\}_{t=1}^T$. At each step:
$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t} x_{t-1}, \beta_t I)$$

2. **Reverse (denoising) process:** A neural network learns to reverse the noising process, gradually denoising from pure Gaussian noise back to a clean sample:
$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \sigma_t^2 I)$$

Using the notation $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$, we can directly compute $x_t$ from $x_0$:
$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

We will use HuggingFace's `diffusers` library to work with pre-trained diffusion models.

### Question 3.1: Understanding the Forward Diffusion Process

Implement the **forward diffusion process** manually:
1. Define a linear beta schedule from $\beta_{start}=0.0001$ to $\beta_{end}=0.02$ over 1000 timesteps.
2. Load a sample MNIST image.
3. Apply forward diffusion at timesteps $t \in \{0, 50, 100, 200, 500, 999\}$.
4. Visualize the progressive noising of the image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets, transforms

def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)

def forward_diffusion(x_0, t, betas):
    # TODO: Compute alphas = 1 - betas
    # TODO: Compute alpha_bars = cumulative product of alphas
    # TODO: Get alpha_bar_t for the specific timestep t
    # TODO: Sample noise epsilon from N(0, I)
    # TODO: Compute x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
    # Return x_t and the noise
    pass

# TODO: Load a sample MNIST image and convert to tensor
# Hint: Use torchvision.datasets.MNIST


# TODO: Define betas using linear_beta_schedule with 1000 timesteps


# TODO: Apply forward diffusion at timesteps [0, 50, 100, 200, 500, 999]
# TODO: Visualize the noised images in a row using matplotlib
timesteps_to_show = [0, 50, 100, 200, 500, 999]


### Question 3.2: Text-to-Image Generation with Stable Diffusion

Use the **SD-Turbo** model (`stabilityai/sd-turbo`) from the HuggingFace `diffusers` library to generate images from text prompts.

1. Load the SD-Turbo pipeline using `AutoPipelineForText2Image`.
2. Generate images for at least 5 different text prompts.
3. Display the generated images in a grid with their prompts as titles.

**Note:** SD-Turbo is a distilled model that generates images in a single step, making it fast and efficient. Use `num_inference_steps=1` and `guidance_scale=0.0` for best results with SD-Turbo.

In [ ]:
from diffusers import AutoPipelineForText2Image
import torch

# Free up memory from previous parts if needed
# del clip_model, vae; torch.cuda.empty_cache()  # uncomment if running low on memory

# Set device and dtype
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# TODO: Load the SD-Turbo pipeline
# Use AutoPipelineForText2Image.from_pretrained("stabilityai/sd-turbo", torch_dtype=dtype)
# Move to device with .to(device)
pipe = None  # TODO

prompts = [
    "A photograph of a golden retriever wearing sunglasses on a beach",
    "A watercolor painting of a cozy cabin in snowy mountains",
    "A futuristic cityscape at sunset with flying cars",
    "A macro photograph of a ladybug on a green leaf",
    "A pencil sketch of an astronaut riding a horse"
]

# TODO: Generate an image for each prompt
# For SD-Turbo use: pipe(prompt, num_inference_steps=1, guidance_scale=0.0).images[0]
generated_images = []  # TODO

# TODO: Display in a grid with titles
# Create a 1x5 subplot grid using matplotlib

### Question 3.3: Exploring the Denoising Process

Visualize how an image evolves during the **reverse (denoising) process** of Stable Diffusion.

Since SD-Turbo only uses 1 step, we will use `stabilityai/stable-diffusion-2-1-base` with a DDIM scheduler and multiple inference steps, capturing intermediate latent states using a **callback function**.

1. Load a Stable Diffusion pipeline with `StableDiffusionPipeline` and set the scheduler to `DDIMScheduler`.
2. Implement a callback function that captures intermediate latent states.
3. Decode the captured latents to images using the pipeline's VAE.
4. Visualize the denoising progression from noise to the final image.

In [ ]:
from diffusers import StableDiffusionPipeline, DDIMScheduler
import torch

# TODO: Load the Stable Diffusion 2.1 base pipeline
# Use StableDiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-2-1-base", torch_dtype=dtype)
# Set scheduler: pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
# Move to device
pipe_multistep = None  # TODO

# Storage for intermediate latents
intermediate_latents = []

def callback_fn(pipe, step_index, timestep, callback_kwargs):
    # TODO: Extract the current latents from callback_kwargs["latents"]
    # Append a detached copy to intermediate_latents
    # Return callback_kwargs
    pass

# TODO: Generate an image with num_inference_steps=20, guidance_scale=7.5, and callback_on_step_end=callback_fn
prompt = "A beautiful landscape painting of mountains at sunrise"


# TODO: Decode selected intermediate latents to images using the pipeline's VAE
# latent_scaled = latent / pipe_multistep.vae.config.scaling_factor
# decoded = pipe_multistep.vae.decode(latent_scaled).sample
# Normalize decoded to [0,1]: (decoded / 2 + 0.5).clamp(0, 1)


# TODO: Visualize the denoising progression
# Display steps 1, 5, 10, 15, 20 in a row

### Question 3.4: Guidance Scale Exploration

**Classifier-free guidance** controls the trade-off between sample quality/prompt adherence and diversity. A higher guidance scale pushes the model to generate images that more closely match the text prompt, but can reduce diversity and introduce artifacts.

1. Use a multi-step Stable Diffusion model (`stabilityai/stable-diffusion-2-1-base` (the pipeline loaded in Q3.3)) with `num_inference_steps=20`.
2. Generate images for the **same prompt** using guidance scale values: `[1, 3, 5, 7.5, 10, 15]`.
3. Display the results side by side with guidance scale values as titles.
4. Use a fixed random seed for fair comparison.

In [ ]:
# TODO: Use the multi-step pipeline loaded in Q3.3 (or load one if not available)

prompt = "A photograph of a cat sitting on a bookshelf surrounded by old books"
guidance_scales = [1, 3, 5, 7.5, 10, 15]
seed = 42

# TODO: For each guidance scale, generate an image with the same seed
# Use torch.Generator(device="cpu").manual_seed(seed) for reproducibility
# Use num_inference_steps=20
images = []

for scale in guidance_scales:
    # TODO: Create a generator with the fixed seed
    # TODO: Generate image with this guidance_scale
    # TODO: Append to images list
    pass

# TODO: Display all images side by side in a single row
# Title each subplot with the guidance scale value
